<a href="https://colab.research.google.com/github/weagan/ARC-AGI-3/blob/main/sarbloh_harness_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sarbloh-Inspired Architecture for ARC-AGI-3
This notebook provides a high-level executable prototype of the **Sarbloh** architecture update designed by Tufa Labs for the ARC-AGI-3 environment. It showcases:
1. A zero-parameter deterministic perception matrix pre-processor.
2. Isolation of static invariant fields (walls) from dynamic variant fields (player/goal).
3. Upgraded Duck Harness REPL providing the model access to the invariant tracker state.
4. Targeted algorithmic code generation to guarantee zero exploratory waste.

### 1. Platform-Agnostic Secret Handling
Loads the required API keys securely across Google Colab and Kaggle platforms.

In [1]:
import os

def load_api_key(secret_name="GEMINI_API_KEY"):
    """Loads a secret securely from Google Colab or Kaggle secret managers."""
    try:
        from google.colab import userdata
        return userdata.get(secret_name)
    except ImportError:
        pass

    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        return user_secrets.get_secret(secret_name)
    except ImportError:
        pass

    return os.environ.get(secret_name, "")

API_KEY = load_api_key()
print("Secret loaded successfully (masked for privacy).")

Secret loaded successfully (masked for privacy).


### 2. Complex Environment & Sarbloh Perception Engine
Instantiates the complex $9 \times 9$ maze and a zero-parameter deterministic processor that splits the raw visual array into tracked objects and static invariants.

In [2]:
import numpy as np

class ComplexArcEnvironment:
    """A larger, complex maze to test pathfinding limits."""
    def __init__(self):
        # 1 = Wall, 0 = Path, 2 = Player, 3 = Goal
        self.grid = np.array([
            [1, 1, 1, 1, 1, 1, 1, 1, 1],
            [1, 2, 0, 1, 0, 0, 0, 0, 1],
            [1, 1, 0, 1, 0, 1, 1, 0, 1],
            [1, 0, 0, 0, 0, 1, 0, 0, 1],
            [1, 0, 1, 1, 1, 1, 0, 1, 1],
            [1, 0, 1, 0, 0, 0, 0, 0, 1],
            [1, 0, 1, 1, 1, 1, 1, 0, 1],
            [1, 0, 0, 0, 0, 0, 1, 3, 1],
            [1, 1, 1, 1, 1, 1, 1, 1, 1]
        ])
        self.player_pos = (1, 1)
        self.moves = 0

    def get_state(self):
        return self.grid.copy()

    def step(self, direction):
        self.moves += 1
        r, c = self.player_pos
        if direction == 'UP': r -= 1
        elif direction == 'DOWN': r += 1
        elif direction == 'LEFT': c -= 1
        elif direction == 'RIGHT': c += 1

        if self.grid[r, c] != 1:
            self.grid[self.player_pos] = 0
            if self.grid[r, c] == 3:
                print("Goal reached autonomously in complex maze!")
            self.player_pos = (r, c)
            self.grid[r, c] = 2
        return self.get_state(), self.moves

class SarblohPerceptionEngine:
    """Deterministic zero-parameter perception tracking engine."""
    @staticmethod
    def extract_layers(grid):
        # Isolate the immutable background structure (walls = 1)
        invariant_mask = (grid == 1).astype(int)

        # Discover key dynamic variants safely
        player_loc = np.argwhere(grid == 2)
        goal_loc = np.argwhere(grid == 3)

        player_pos = tuple(player_loc[0]) if len(player_loc) > 0 else None
        goal_pos = tuple(goal_loc[0]) if len(goal_loc) > 0 else None

        return {
            "invariant_field": invariant_mask.tolist(),
            "player_pos": player_pos,
            "goal_pos": goal_pos
        }

### 3. Upgraded Duck Harness (Sarbloh Integration)
Exposes the extracted invariant fields and tracking layers directly to the local REPL environment as context state properties.

In [3]:
import sys
from io import StringIO

class UpgradedDuckHarness:
    def __init__(self):
        self.perception_tracker = None

    def update_perception(self, grid):
        self.perception_tracker = SarblohPerceptionEngine.extract_layers(grid)

    def get_invariant_matrix(self):
        return self.perception_tracker["invariant_field"]

    def execute_code_repl(self, code_str, env):
        old_stdout = sys.stdout
        redirected_output = StringIO()
        sys.stdout = redirected_output

        # Expose both the environment and harness properties into the REPL
        local_vars = {"env": env, "harness": self}
        try:
            exec(code_str, {}, local_vars)
            error = None
        except Exception as e:
            error = str(e)

        sys.stdout = old_stdout
        return redirected_output.getvalue(), error

### 4. Running the Few-Shot Transformation Demonstration
Feeds Gemini the spatial pairs alongside the architectural instructions to write algorithmic graph searches inside the sandbox instead of guessing coordinates.

In [4]:
from google import genai
import re

client = genai.Client(api_key=API_KEY)

env = ComplexArcEnvironment()
harness = UpgradedDuckHarness()
harness.update_perception(env.get_state())

metadata_summary = f"""
Perception Summary:
- Invariant Field Array dimensions: 9x9 (1 indicates structural boundaries)
- Active Player ('2') coordinate: {harness.perception_tracker['player_pos']}
- Active Goal ('3') coordinate: {harness.perception_tracker['goal_pos']}
"""

sarbloh_prompt = f"""
You are an AI engineer writing code inside an active execution harness layer to solve an ARC grid puzzle.

Background Engine Information:
{metadata_summary}

Available Interface Methods:
- harness.get_invariant_matrix(): returns a list-of-lists grid layer where 1 represents non-traversable walls.
- env.player_pos: returns current tuple position (row, col) of the agent.
- env.step(direction): moves player ('UP', 'DOWN', 'LEFT', 'RIGHT')

Task Instruction:
Write a concise Python pathfinding implementation (such as BFS or DFS) to navigate the player from its current start coordinates to the goal coordinates using 'harness.get_invariant_matrix()' to check for wall constraints. Then loop through the calculated path executing the necessary 'env.step()' methods.

Return ONLY the executable Python code block inside a markdown sequence. Do not provide descriptive conversational text outside the block.
"""

if not API_KEY:
    print("[Notice] No API Key found. Emulating the Sarbloh algorithmic script execution block fallback...")
    # Fallback to structural pathfinder logic mirroring optimal model algorithmic output
    live_code_block = """
from collections import deque

def find_path(start, goal, walls):
    queue = deque([[start]])
    visited = {start}

    while queue:
        path = queue.popleft()
        r, c = path[-1]
        if (r, c) == goal:
            return path

        for d, (dr, dc) in [('UP', -1, 0), ('DOWN', 1, 0), ('LEFT', 0, -1), ('RIGHT', 0, 1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < 9 and 0 <= nc < 9 and walls[nr][nc] != 1 and (nr, nc) not in visited:
                visited.add((nr, nc))
                queue.append(path + [(nr, nc, d)])
    return []

walls = harness.get_invariant_matrix()
start = env.player_pos
goal = (7, 7)

calculated_route = find_path(start, goal, walls)
for step_node in calculated_route:
    if len(step_node) == 3:
        env.step(step_node[2])
"""
else:
    print("--- Communicating with Gemini API to fetch dynamic pathfinding script... ---")
    response = client.models.generate_content(model='gemini-2.5-flash', contents=sarbloh_prompt)
    blocks = re.findall(r'```python\s*(.*?)\s*```', response.text, re.DOTALL)
    live_code_block = blocks[0] if blocks else response.text

print("\n--- Executable Script Passed into REPL Sandbox ---")
print(live_code_block)

output, err = harness.execute_code_repl(live_code_block, env)
if err:
    print(f"\nExecution Error: {err}")
else:
    print(f"\n--- Harness Evaluation Log ---\n{output if output else 'Pipeline ran cleanly with zero constraint errors.'}")

print("\n--- Final Grid Verification ---")
print("\n".join([" ".join(map(str, row)) for row in env.get_state()]))

--- Communicating with Gemini API to fetch dynamic pathfinding script... ---

--- Executable Script Passed into REPL Sandbox ---
import collections
import numpy as np

def solve_arc_puzzle():
    grid = harness.get_invariant_matrix()
    rows = len(grid)
    cols = len(grid[0])

    start_pos = env.player_pos
    goal_pos = (np.int64(7), np.int64(7))

    # BFS implementation
    queue = collections.deque([(start_pos, [])])  # Stores (current_position, path_taken_to_reach_here)
    visited = {start_pos}
    
    # Define possible movements and their corresponding string directions
    # (delta_row, delta_col, direction_string)
    moves = [
        (-1, 0, 'UP'),
        (1, 0, 'DOWN'),
        (0, -1, 'LEFT'),
        (0, 1, 'RIGHT'),
    ]

    path_to_goal = []

    while queue:
        current_pos, current_path = queue.popleft()

        if current_pos == goal_pos:
            path_to_goal = current_path
            break

        r, c = current_pos

        for dr, dc, direction_s